Update Holiday Calendar
Fetches Sri Lankan public holidays for the current year, and for next year once it's August 1st or later (so next year's calendar is ready ahead of time). Merges with existing saved calendar.

**Output**: gold/erp/reference/holiday_calendar.parquet

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, save_gold, read_gold
from src.reference_data.holiday_calendar import fetch_holidays
import pandas as pd
import datetime

blob_service = get_blob_service(storage_account_name, storage_account_key)

Determine which years to fetch

In [0]:
today = datetime.date.today()
years_to_fetch = [today.year]

# From August 1st onward, also fetch next year's calendar
if today.month >= 8:
    years_to_fetch.append(today.year + 1)

print(f"Fetching holidays for: {years_to_fetch}")

Fetch and merge with existing calendar

In [0]:
new_years_df = pd.concat(
    [fetch_holidays(y, holiday_api_base_url, holiday_api_token) for y in years_to_fetch],
    ignore_index=True
)

try:
    existing = read_gold(blob_service, "live/battery/data/reference/holiday_calendar.parquet")
    if existing.empty:
            combined = new_years_df.copy()
    else:
            combined = pd.concat([existing, new_years_df], ignore_index=True)
except Exception:
        combined = new_years_df.copy()

combined = combined.drop_duplicates(subset="date", keep="last")
combined = combined.sort_values("date")

print(f"Total holidays in calendar: {len(combined)}")
print(combined.tail(10))

save_gold(blob_service, combined, "live/battery/data/reference/holiday_calendar.parquet")
print("Saved holiday_calendar.parquet")